# Example 2.4 / 4: Data from Google Sheets

Companion notebook for **2.4 Data from pandas, NumPy and Files** in the AMPL lab Documentation, section "Data from Google Sheets". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Data from Google Sheets

Success is a list of rows whose first row holds the column headers, such as
`[["FOOD", "cost"], ["BEEF", 3.59], ...]`. `UNFORMATTED_VALUE` returns numbers as numbers rather
than as the formatted text shown in the sheet. `gclient.open("DietModelSheet")` opens a sheet by
its name instead of its URL.

From there the rows are ordinary Python lists, and the rest runs anywhere. The AMPL Colab
notebooks use two helpers: `table_to_dataframe` makes the first column the index, and
`matrix_to_dataframe` turns a matrix laid out with one row per food and one column per nutrient
into the long, two-level form that `set_data()` expects. With `tr=True` it swaps the two labels,
so a sheet with foods down the side fills `amt {NUTR, FOOD}`. Here the rows are typed in, in the
shape `get_values()` returns:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FOOD;                  # foods, from the first column of sheet FOOD
param cost {FOOD} > 0;     # cost per package
set NUTR;                  # nutrients
param amt {NUTR, FOOD} >= 0;  # nutrient per package; sheet has foods as rows

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

def table_to_dataframe(rows):
    return pd.DataFrame(rows[1:], columns=rows[0]).set_index(rows[0][0])

def matrix_to_dataframe(rows, tr=False):
    col_labels = rows[0][1:]
    row_labels = [row[0] for row in rows[1:]]
    def label(pair):
        return pair if not tr else (pair[1], pair[0])
    data = {label((r, c)): rows[i + 1][j + 1]
            for i, r in enumerate(row_labels) for j, c in enumerate(col_labels)}
    df = pd.Series(data).reset_index()
    df.columns = ["index1", "index2", rows[0][0]]
    return df.set_index(["index1", "index2"])

food_rows = [["FOOD", "cost"], ["BEEF", 3.59], ["CHK", 2.59]]
amt_rows = [["amt", "A", "C"], ["BEEF", 60, 20], ["CHK", 8, 0]]

ampl.set_data(table_to_dataframe(food_rows), "FOOD")
ampl.set["NUTR"] = amt_rows[0][1:]
ampl.set_data(matrix_to_dataframe(amt_rows, tr=True))

In [5]:
# run.py
print(ampl.param["cost"].to_dict())
print(ampl.param["amt"].to_dict())

{'BEEF': 3.59, 'CHK': 2.59}
{('A', 'BEEF'): 60, ('A', 'CHK'): 8, ('C', 'BEEF'): 20, ('C', 'CHK'): 0}


### Pure AMPL: data.dat and model.run

The same instance in AMPL data syntax; the `(tr)` marker reads a table laid out with foods as
rows into `amt {NUTR, FOOD}`:

In [6]:
%%writefile data.dat
set FOOD := BEEF CHK;
param cost := BEEF 3.59  CHK 2.59;
set NUTR := A C;
param amt (tr):  A   C :=
  BEEF          60  20
  CHK            8   0 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display cost;
display amt;

cost [*] :=
BEEF  3.59
 CHK  2.59
;

amt :=
A BEEF   60
A CHK     8
C BEEF   20
C CHK     0
;

